In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.4 Multi-head attention

One attention is one relationship. Multi-head runs several in parallel on
slices of the channels, so different heads can track different patterns (syntax,
subject agreement, position), then concatenates them back.

In [ ]:
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
tokens, n_head, hs = 5, 4, 6
channels = n_head * hs

def to_heads(t):
    return t.view(1, tokens, n_head, hs).transpose(1, 2)   # [1, n_head, T, hs]

q = to_heads(torch.randn(1, tokens, channels))
k = to_heads(torch.randn(1, tokens, channels))
v = to_heads(torch.randn(1, tokens, channels))
out, w = scaled_dot_product_attention(q, k, v, causal=True)
combined = out.transpose(1, 2).contiguous().view(1, tokens, channels)
print("per-head out:", tuple(out.shape), " recombined:", tuple(combined.shape))

The recombined tensor is back to `[batch, tokens, channels]`, ready for the
next layer. The heads were a parallel detour, invisible from outside the block.

In [ ]:
assert tuple(out.shape) == (1, n_head, tokens, hs)
assert tuple(w.shape) == (1, n_head, tokens, tokens)
assert tuple(combined.shape) == (1, tokens, channels)